In [1]:
import pandas as pd
import pyfixest as pf

panel = pd.read_csv("../data/processed/panel.csv")
d = panel[panel["main_sample"]].dropna(subset=["ln_s_may", "ln_e_2025"]).copy()
d["country_id"] = d["geo"].astype("category").cat.codes

In [2]:
ols = pf.feols("ln_s_may ~ ln_e_2025 | geo + isco2", data=d, vcov={"CRV1": "country_id"})
ols.summary()

###

Estimation:  OLS
Dep. var.: ln_s_may, Fixed effects: geo + isco2
sample: None = all
Inference:  CRV1
Observations:  482

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| ln_e_2025     |      0.036 |        0.035 |     1.018 |      0.325 | -0.039 |   0.111 |
---
RMSE: 0.221 R2: 0.982 R2 Within: 0.005 


In [3]:
d["gap"] = d["ln_s_may"] - d["ln_e_2025"]
ols_vs1 = pf.feols("gap ~ ln_e_2025 | geo + isco2", data=d, vcov={"CRV1": "country_id"})
ols_vs1.summary()

###

Estimation:  OLS
Dep. var.: gap, Fixed effects: geo + isco2
sample: None = all
Inference:  CRV1
Observations:  482

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| ln_e_2025     |     -0.964 |        0.035 |   -27.423 |      0.000 | -1.039 |  -0.889 |
---
RMSE: 0.221 R2: 0.979 R2 Within: 0.785 


In [4]:
print(ols.wildboottest(param="ln_e_2025", reps=9999, seed=42))
print(ols_vs1.wildboottest(param="ln_e_2025", reps=9999, seed=42))

param                       ln_e_2025
t value            1.0002538151586442
Pr(>|t|)          0.36483648364836485
bootstrap_type                     11
inference             CRV(country_id)
impose_null                      True
ssc                          1.193178
dtype: object
param                      ln_e_2025
t value           -26.95714516598611
Pr(>|t|)                         0.0
bootstrap_type                    11
inference            CRV(country_id)
impose_null                     True
ssc                         1.193178
dtype: object


In [5]:
iv = pf.feols("ln_s_may ~ 1 | geo + isco2 | ln_e_2025 ~ ln_e_2015", data=d, vcov={"CRV1": "country_id"})
iv.summary()
print("First-stage F:", iv._f_stat_1st_stage)

###

Estimation:  IV
Dep. var.: ln_s_may, Fixed effects: geo + isco2
sample: None = all
Inference:  CRV1
Observations:  482

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| ln_e_2025     |      0.020 |        0.038 |     0.529 |      0.604 | -0.061 |   0.102 |
---

First-stage F: 159.48812525029138


In [6]:
ppml = pf.fepois("s_may ~ ln_e_2025 | geo + isco2", data=d, vcov={"CRV1": "country_id"})
ppml.summary()

###

Estimation:  Poisson
Dep. var.: s_may, Fixed effects: geo + isco2
sample: None = all
Inference:  CRV1
Observations:  482

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| ln_e_2025     |     -0.010 |        0.022 |    -0.459 |      0.646 | -0.053 |   0.033 |
---
Deviance: 0.216 
